# Faza 1 — Analiza feature space klasyfikatora Real vs AI

## Cel

Sprawdzić, **czy w przestrzeni cech wytrenowanego klasyfikatora EfficientNetB0 istnieje sensowny kierunek "AI-ness"** — taki, że przesunięcie wzdłuż niego spójnie oddziela obrazy AI od Real, i którym da się manipulować.

Jeśli odpowiedź brzmi TAK, ma sens budować Fazę 2 (VAE z dekoderem do edycji obrazu). Jeśli NIE, trzeba zmienić podejście.

## Co robimy

1. **Cache pipeline** — surowe obrazy → feature vectors → wytrenowane SVM/UMAP, każdy etap zapisany na dysk
2. **Dwie warstwy feature space** — porównanie GAP (1280D, surowa) vs Dense (256D, blisko decyzji)
3. **UMAP** — wizualizacja 2D, kolorowanie per generator
4. **Trzy metody znajdowania kierunku AI** — różnica średnich, SVM, Logistic Regression
5. **Adversarial test** — czy przesunięcie feature vectora wzdłuż kierunku faktycznie zmienia decyzję klasyfikatora?
6. **Per-generator analiza** — czy SD, DALL-E, Midjourney mają wspólny czy różne kierunki AI-ness?

## Wymagania

- Wytrenowany model `saved_models/best_model.keras` z `defactify_training_v3.ipynb`
- Dostęp do datasetu `Rajarshi-Roy-research/Defactify_Image_Dataset`
- ~3 GB miejsca na cache dla pełnego datasetu
- ~1h GPU lub kilka godzin CPU dla pierwszego runu


## 0. Setup i strategia cachingu


In [ ]:
# !pip install -q umap-learn scikit-learn matplotlib


In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# ─── Stałe ───
IMG_SIZE = 224
BATCH_SIZE = 64
MODEL_PATH = "saved_models/best_model.keras"
CACHE_DIR = Path("cache_phase1")
CACHE_DIR.mkdir(exist_ok=True)

LABEL_A_MAP = {0: "Real", 1: "AI-Generated"}
LABEL_B_MAP = {
    0: "Real", 1: "SD 2.1", 2: "SD XL",
    3: "SD 3", 4: "DALL-E 3", 5: "Midjourney 6"
}

print(f"\nCache directory: {CACHE_DIR.resolve()}")


In [ ]:
# ─── Helpers do cachowania ───
# Każda kosztowna operacja jest zapisywana na dysk po pierwszym wyliczeniu.
# Drugi run notebooka ładuje z cache w ułamku sekundy.

def cache_npy(path, compute_fn, force=False):
    """Cache numpy array. Computes & saves on first run, loads after."""
    path = Path(path)
    if path.exists() and not force:
        print(f"  ↳ Loading from cache: {path.name}")
        return np.load(path)
    print(f"  ↳ Computing {path.name} (will be cached)...")
    arr = compute_fn()
    np.save(path, arr)
    print(f"     ✅ Saved: {path.name} ({arr.nbytes / 1e6:.1f} MB)")
    return arr

def cache_pickle(path, compute_fn, force=False):
    """Cache arbitrary Python object via pickle."""
    path = Path(path)
    if path.exists() and not force:
        print(f"  ↳ Loading from cache: {path.name}")
        with open(path, 'rb') as f:
            return pickle.load(f)
    print(f"  ↳ Computing {path.name} (will be cached)...")
    obj = compute_fn()
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"     ✅ Saved: {path.name}")
    return obj

def cache_npz(path, compute_fn, force=False):
    """Cache dict-of-arrays via np.savez_compressed."""
    path = Path(path)
    if path.exists() and not force:
        print(f"  ↳ Loading from cache: {path.name}")
        return dict(np.load(path))
    print(f"  ↳ Computing {path.name} (will be cached)...")
    d = compute_fn()
    np.savez_compressed(path, **d)
    print(f"     ✅ Saved: {path.name}")
    return d

print("Cache helpers ready ✅")


## 1. Załaduj dataset i zachowaj indeksy + metadane (cache 1)

Nie cachujemy surowych obrazów (zbyt duże dla pełnego datasetu — kilkadziesiąt GB), bo i tak musimy przepuścić je przez model. Cachujemy tylko metadane (indeksy, etykiety) żeby pipeline był deterministyczny.


In [ ]:
from datasets import load_dataset

ds = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset")
train = ds['train']
print(f"Total samples: {len(train):,}")

# Cache: etykiety jako numpy arrays (szybki dostęp bez iterowania HF)
def _compute_labels():
    label_a = np.array(train['Label_A'], dtype=np.int8)
    label_b = np.array(train['Label_B'], dtype=np.int8)
    return {'label_a': label_a, 'label_b': label_b}

labels_cache = cache_npz(CACHE_DIR / "labels.npz", _compute_labels)
label_a = labels_cache['label_a']
label_b = labels_cache['label_b']

n_real = (label_a == 0).sum()
n_ai = (label_a == 1).sum()
print(f"Real: {n_real:,}  AI: {n_ai:,}")
print(f"Per generator (Label_B):")
for k, v in LABEL_B_MAP.items():
    count = (label_b == k).sum()
    print(f"  {k} ({v}): {count:,}")


## 2. Załaduj model i wyciągnij dwa feature extractors

Robimy to *raz*, potem reużywamy. Definiujemy dwa modele pomocnicze:
- **GAP feature extractor** — wyjście z `GlobalAveragePooling2D` (1280D, surowa reprezentacja z backbone)
- **Dense feature extractor** — wyjście z `Dense(256, relu)` (skondensowana, blisko decyzji)

Plus zachowujemy **head** (warstwy po Dense(256)) żeby móc puszczać przesunięte feature vectors przez końcówkę modelu w teście adversarial.


In [ ]:
from tensorflow.keras import models, layers

model = tf.keras.models.load_model(MODEL_PATH)
model.trainable = False
print(f"Loaded: {MODEL_PATH}")
model.summary()


In [ ]:
# ─── Znajdź kluczowe warstwy ───
# Z notebooka treningowego architektura: input → EfficientNetB0 → GAP → Dropout → Dense(256) → Dropout → Dense(1, sigmoid)

gap_layer = None
dense_256_layer = None
dense_out_layer = None

for layer in model.layers:
    if isinstance(layer, layers.GlobalAveragePooling2D):
        gap_layer = layer
    elif isinstance(layer, layers.Dense):
        if layer.units == 256:
            dense_256_layer = layer
        elif layer.units == 1:
            dense_out_layer = layer

assert gap_layer is not None, "Nie znaleziono warstwy GAP"
assert dense_256_layer is not None, "Nie znaleziono Dense(256)"
assert dense_out_layer is not None, "Nie znaleziono Dense(1)"

print(f"GAP layer:        {gap_layer.name}, output shape: {gap_layer.output.shape}")
print(f"Dense(256) layer: {dense_256_layer.name}, output shape: {dense_256_layer.output.shape}")
print(f"Output layer:     {dense_out_layer.name}, output shape: {dense_out_layer.output.shape}")

# ─── Feature extractors ───
feat_gap = models.Model(model.inputs, gap_layer.output, name='feat_gap')
feat_dense = models.Model(model.inputs, dense_256_layer.output, name='feat_dense')

# ─── Head: GAP → output (do testu adversarial) ───
# Bierzemy feature vector na poziomie GAP i przepuszczamy przez resztę sieci
head_input = layers.Input(shape=(1280,), name='gap_feature_input')
x = head_input
# Replikujemy warstwy po GAP — kolejność z notebooka: Dropout → Dense(256) → Dropout → Dense(1)
head_started = False
for layer in model.layers:
    if layer is gap_layer:
        head_started = True
        continue
    if not head_started:
        continue
    # Skopiuj warstwę do head — używamy istniejących wag (tej samej referencji)
    x = layer(x)

head_from_gap = models.Model(head_input, x, name='head_from_gap')
head_from_gap.trainable = False

print(f"\nFeature extractors gotowe:")
print(f"  feat_gap:      input → 1280D")
print(f"  feat_dense:    input → 256D")
print(f"  head_from_gap: 1280D → 1D (sigmoid)")


## 3. Wyciągnij feature vectors z całego datasetu (cache 2 — najbardziej kosztowny)

**To jest najdłuższy krok** — pełny dataset, dwie warstwy = ~1h GPU lub kilka godzin CPU.

Po wykonaniu raz: kolejne runy notebooka ładują z cache w sekundach. Liczymy obie warstwy w jednym przejściu przez dataset (nie chcemy 2x iterować HuggingFace).

Cachujemy **dwa pliki** (`features_gap.npy`, `features_dense.npy`), każdy ~ liczba próbek × wymiar × 4 bajty.


In [ ]:
from PIL import Image
import time

FEAT_GAP_PATH = CACHE_DIR / "features_gap.npy"
FEAT_DENSE_PATH = CACHE_DIR / "features_dense.npy"

def _compute_features():
    """Iteruje cały dataset, wyciąga features z obu warstw jednocześnie.
    
    Strategia: budujemy mini-model z 2 wyjściami (GAP + Dense), żeby jednym
    forward pass dostać oba wektory. Oszczędza ~50% czasu vs dwa oddzielne predicty.
    """
    multi_output = models.Model(
        model.inputs,
        [gap_layer.output, dense_256_layer.output],
        name='multi_output_extractor'
    )
    multi_output.trainable = False

    n = len(train)
    feats_gap = np.zeros((n, 1280), dtype=np.float32)
    feats_dense = np.zeros((n, 256), dtype=np.float32)

    batch_imgs = []
    batch_indices = []
    start = time.time()

    def flush_batch():
        if not batch_imgs:
            return
        x = np.stack(batch_imgs).astype(np.float32)  # [0, 255], EfficientNet normalizuje sam
        gap_out, dense_out = multi_output.predict(x, verbose=0)
        for i, idx in enumerate(batch_indices):
            feats_gap[idx] = gap_out[i]
            feats_dense[idx] = dense_out[i]
        batch_imgs.clear()
        batch_indices.clear()

    for i in range(n):
        sample = train[i]
        img = sample['Image'].convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        batch_imgs.append(np.array(img, dtype=np.float32))
        batch_indices.append(i)
        if len(batch_imgs) >= BATCH_SIZE:
            flush_batch()
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - start
            eta = elapsed / (i + 1) * (n - i - 1)
            print(f"  {i+1:,}/{n:,} ({100*(i+1)/n:.1f}%) elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")
    flush_batch()

    elapsed = time.time() - start
    print(f"  ✅ Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
    return feats_gap, feats_dense

# Cache wykonujemy raz; przy kolejnych runach ładujemy z dysku
if FEAT_GAP_PATH.exists() and FEAT_DENSE_PATH.exists():
    print("  ↳ Loading features from cache...")
    features_gap = np.load(FEAT_GAP_PATH)
    features_dense = np.load(FEAT_DENSE_PATH)
    print(f"  ✅ Loaded: GAP {features_gap.shape}, Dense {features_dense.shape}")
else:
    print("  ↳ Computing features (this is the slow step — go get coffee)...")
    features_gap, features_dense = _compute_features()
    np.save(FEAT_GAP_PATH, features_gap)
    np.save(FEAT_DENSE_PATH, features_dense)
    print(f"  ✅ Saved: GAP {features_gap.shape}, Dense {features_dense.shape}")

print(f"\nFeatures GAP:   shape={features_gap.shape}, dtype={features_gap.dtype}")
print(f"Features Dense: shape={features_dense.shape}, dtype={features_dense.dtype}")
print(f"GAP total size:   {features_gap.nbytes / 1e6:.0f} MB")
print(f"Dense total size: {features_dense.nbytes / 1e6:.0f} MB")


## 4. UMAP — wizualizacja struktury feature space (cache 3)

UMAP redukuje 1280D / 256D do 2D zachowując strukturę lokalną (klastry) i częściowo globalną.

**Co chcemy zobaczyć:**
- Czy Real i AI tworzą rozdzielone klastry? Jak czyste są granice?
- Czy poszczególne generatory (SD, DALL-E, Midjourney) tworzą sub-klastry wewnątrz "AI"?
- Czy GAP vs Dense pokazują różną strukturę? (Dense powinien być bardziej skondensowany — został wytrenowany pod separację)

UMAP jest deterministyczny przy ustawionym `random_state`. Cachujemy wytrenowany reducer + 2D embedding.


In [ ]:
import umap

UMAP_GAP_EMB = CACHE_DIR / "umap_gap_embedding.npy"
UMAP_DENSE_EMB = CACHE_DIR / "umap_dense_embedding.npy"
UMAP_GAP_REDUCER = CACHE_DIR / "umap_gap_reducer.pkl"
UMAP_DENSE_REDUCER = CACHE_DIR / "umap_dense_reducer.pkl"

def fit_umap(features, name):
    """Fituje UMAP z parametrami zoptymalizowanymi pod wizualizację klastrów."""
    print(f"  Fitting UMAP on {name} ({features.shape})...")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=30,       # więcej = bardziej globalna struktura
        min_dist=0.1,         # mniej = bardziej zwarte klastry
        metric='cosine',      # cosine lepiej dla feature wektorów z ReLU
        random_state=42,
        verbose=True,
    )
    embedding = reducer.fit_transform(features)
    return reducer, embedding

# ── GAP ──
if UMAP_GAP_EMB.exists() and UMAP_GAP_REDUCER.exists():
    print("  ↳ Loading UMAP GAP from cache...")
    emb_gap = np.load(UMAP_GAP_EMB)
    with open(UMAP_GAP_REDUCER, 'rb') as f:
        reducer_gap = pickle.load(f)
else:
    reducer_gap, emb_gap = fit_umap(features_gap, "GAP (1280D)")
    np.save(UMAP_GAP_EMB, emb_gap)
    with open(UMAP_GAP_REDUCER, 'wb') as f:
        pickle.dump(reducer_gap, f)

# ── Dense ──
if UMAP_DENSE_EMB.exists() and UMAP_DENSE_REDUCER.exists():
    print("  ↳ Loading UMAP Dense from cache...")
    emb_dense = np.load(UMAP_DENSE_EMB)
    with open(UMAP_DENSE_REDUCER, 'rb') as f:
        reducer_dense = pickle.load(f)
else:
    reducer_dense, emb_dense = fit_umap(features_dense, "Dense (256D)")
    np.save(UMAP_DENSE_EMB, emb_dense)
    with open(UMAP_DENSE_REDUCER, 'wb') as f:
        pickle.dump(reducer_dense, f)

print(f"\nEmbeddings: GAP {emb_gap.shape}, Dense {emb_dense.shape}")


In [ ]:
# ─── Wizualizacja: 2x2 grid ───
# Górny rząd: kolorowanie binarne (Real vs AI)
# Dolny rząd: kolorowanie per generator

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Helper do binarnej wizualizacji
def plot_binary(ax, emb, title):
    real_mask = label_a == 0
    ai_mask = label_a == 1
    ax.scatter(emb[real_mask, 0], emb[real_mask, 1],
               c='#1f77b4', s=2, alpha=0.4, label=f'Real ({real_mask.sum():,})')
    ax.scatter(emb[ai_mask, 0], emb[ai_mask, 1],
               c='#d62728', s=2, alpha=0.4, label=f'AI ({ai_mask.sum():,})')
    ax.set_title(title, fontsize=13)
    ax.legend(markerscale=4, loc='best')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.grid(alpha=0.2)

# Helper do per-generator wizualizacji
def plot_per_generator(ax, emb, title):
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for k in range(6):
        mask = label_b == k
        if mask.sum() == 0:
            continue
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=colors[k], s=2, alpha=0.5,
                   label=f'{LABEL_B_MAP[k]} ({mask.sum():,})')
    ax.set_title(title, fontsize=13)
    ax.legend(markerscale=4, loc='best', fontsize=9)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.grid(alpha=0.2)

plot_binary(axes[0, 0], emb_gap, 'GAP (1280D) — Real vs AI')
plot_binary(axes[0, 1], emb_dense, 'Dense (256D) — Real vs AI')
plot_per_generator(axes[1, 0], emb_gap, 'GAP (1280D) — per generator')
plot_per_generator(axes[1, 1], emb_dense, 'Dense (256D) — per generator')

plt.tight_layout()
plt.savefig(CACHE_DIR / 'umap_overview.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nCo szukać na obrazkach:")
print("  • Czy Real i AI są rozdzielone na lewym/prawym wykresie?")
print("  • Czy Dense (kolumna 2) ma czystszą separację niż GAP (kolumna 1)?")
print("  • Czy generatory tworzą sub-klastry (dolny rząd)?")
print("  • Czy są generatory bliżej/dalej od domeny Real?")


## 5. Trzy metody znajdowania kierunku "AI-ness"

Pełna pętla dla obu warstw (GAP i Dense) i trzech metod:
1. **Różnica średnich** — najprostsza: `μ_AI - μ_Real`. Działa, jeśli chmury są w przybliżeniu gaussowskie i rozdzielone.
2. **SVM (LinearSVC)** — wektor normalny do hiperpłaszczyzny separującej. Bierze pod uwagę kształt rozkładów, nie tylko środki.
3. **Logistic Regression** — podobnie do SVM, ale daje też prawdopodobieństwa.

Porównanie: **cosine similarity między kierunkami**. Jeśli wszystkie trzy metody dają podobne kierunki (cos > 0.9), kierunek "AI-ness" jest robustny.


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

DIRECTIONS_PATH = CACHE_DIR / "ai_directions.pkl"
MODELS_PATH = CACHE_DIR / "sklearn_models.pkl"

def normalize(v):
    return v / np.linalg.norm(v)

def compute_directions(features, labels, name):
    """Liczy 3 kierunki AI dla danej przestrzeni cech + zwraca wytrenowane modele."""
    print(f"\n--- {name} (dim={features.shape[1]}) ---")

    # Train/test split — modele trenujemy na 80%, walidujemy na 20%
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42, stratify=labels
    )

    # 1. Różnica średnich
    mu_real = features[labels == 0].mean(axis=0)
    mu_ai = features[labels == 1].mean(axis=0)
    dir_mean = normalize(mu_ai - mu_real)

    # 2. SVM
    print(f"  Training LinearSVC...")
    svm = LinearSVC(C=1.0, max_iter=5000, dual='auto')
    svm.fit(X_train, y_train)
    dir_svm = normalize(svm.coef_[0])
    svm_acc = svm.score(X_test, y_test)

    # 3. Logistic Regression
    print(f"  Training LogisticRegression...")
    lr = LogisticRegression(C=1.0, max_iter=5000)
    lr.fit(X_train, y_train)
    dir_lr = normalize(lr.coef_[0])
    lr_acc = lr.score(X_test, y_test)

    print(f"  SVM test accuracy: {svm_acc:.4f}")
    print(f"  LR  test accuracy: {lr_acc:.4f}")

    # Cosine similarity między kierunkami
    print(f"\n  Cosine similarity between directions:")
    print(f"    mean ↔ svm: {np.dot(dir_mean, dir_svm):+.4f}")
    print(f"    mean ↔ lr:  {np.dot(dir_mean, dir_lr):+.4f}")
    print(f"    svm ↔ lr:   {np.dot(dir_svm, dir_lr):+.4f}")

    return {
        'mean': dir_mean,
        'svm': dir_svm,
        'lr': dir_lr,
        'mu_real': mu_real,
        'mu_ai': mu_ai,
        'svm_acc': svm_acc,
        'lr_acc': lr_acc,
    }, {'svm': svm, 'lr': lr}

if DIRECTIONS_PATH.exists() and MODELS_PATH.exists():
    print("  ↳ Loading directions from cache...")
    with open(DIRECTIONS_PATH, 'rb') as f:
        directions = pickle.load(f)
    with open(MODELS_PATH, 'rb') as f:
        sklearn_models = pickle.load(f)
else:
    dirs_gap, models_gap = compute_directions(features_gap, label_a, "GAP")
    dirs_dense, models_dense = compute_directions(features_dense, label_a, "Dense")
    directions = {'gap': dirs_gap, 'dense': dirs_dense}
    sklearn_models = {'gap': models_gap, 'dense': models_dense}
    with open(DIRECTIONS_PATH, 'wb') as f:
        pickle.dump(directions, f)
    with open(MODELS_PATH, 'wb') as f:
        pickle.dump(sklearn_models, f)

print("\n" + "="*60)
print("Wszystkie kierunki AI policzone i zachowane ✅")


In [ ]:
# ─── Wizualizacja: histogramy projekcji na kierunki ───
# Jeśli kierunek jest dobry, projekcja Real i AI powinna być rozdzielona

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (space_name, feats) in enumerate([('GAP', features_gap), ('Dense', features_dense)]):
    space_key = space_name.lower()
    for col, method in enumerate(['mean', 'svm', 'lr']):
        direction = directions[space_key][method]
        proj = feats @ direction
        proj_real = proj[label_a == 0]
        proj_ai = proj[label_a == 1]

        ax = axes[row, col]
        ax.hist(proj_real, bins=80, alpha=0.6, color='#1f77b4', label='Real', density=True)
        ax.hist(proj_ai, bins=80, alpha=0.6, color='#d62728', label='AI', density=True)
        
        # Dodaj statystyki separacji
        d_prime = (proj_ai.mean() - proj_real.mean()) / np.sqrt(0.5 * (proj_ai.var() + proj_real.var()))
        ax.set_title(f"{space_name} — {method.upper()}  (d' = {d_prime:.2f})", fontsize=12)
        ax.set_xlabel('Projection onto AI direction')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(CACHE_DIR / 'projection_histograms.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nd' (Cohen's d) — miara separacji rozkładów:")
print("  d' < 1:  słaba separacja")
print("  d' 1–2:  umiarkowana — kierunek istnieje, ale klasy się nakładają")
print("  d' > 2:  silna separacja — kierunek 'AI-ness' jest mocny")
print("  d' > 3:  bardzo silna — klasy prawie nie nakładają się")


## 6. Adversarial test — czy kierunek faktycznie steruje decyzją modelu?

**Najważniejsza walidacja całej Fazy 1.**

Procedura: bierzemy każdy obraz AI z testowej próbki, wyciągamy jego feature vector na poziomie GAP, przesuwamy go o `α · ai_direction` w stronę Real (z różnymi α), przepuszczamy przez resztę sieci (Dropout → Dense(256) → Dropout → Dense(1, sigmoid)) i patrzymy, czy `prob_AI` spada.

**Co robimy tylko na warstwie GAP** (1280D)? Bo head modelu (Dense → Dense → sigmoid) operuje na wyjściu GAP. Edycja na poziomie Dense(256) byłaby "za późno" — manipulujemy tylko ostatnie 2 warstwy.

**Czego oczekujemy:**
- α = 0: prob_AI bliskie oryginalnego (czyli wysokie dla AI obrazów)
- α rosnące: prob_AI monotonicznie spada
- Przy pewnym α prob_AI < 0.5 → model zmienił decyzję
- Wszystkie trzy kierunki (mean/svm/lr) powinny działać, choć z różną "siłą"


In [ ]:
ADVERSARIAL_PATH = CACHE_DIR / "adversarial_test.npz"

# Bierzemy próbkę AI obrazów do testu (zachowujemy reszty na ewaluację)
ai_indices = np.where(label_a == 1)[0]
real_indices = np.where(label_a == 0)[0]
rng = np.random.RandomState(42)

# 2000 AI obrazów do testu — wystarczająco do solidnych statystyk
test_ai_idx = rng.choice(ai_indices, size=min(2000, len(ai_indices)), replace=False)
test_real_idx = rng.choice(real_indices, size=min(2000, len(real_indices)), replace=False)

feats_ai_test = features_gap[test_ai_idx]
feats_real_test = features_gap[test_real_idx]

# Zakres alpha — od 0 (brak edycji) do dużych wartości (silne pchnięcie)
# Skalujemy do typowego rzędu wielkości feature wektorów
feat_norm_typical = np.linalg.norm(feats_ai_test, axis=1).mean()
alphas = np.linspace(0, 3 * feat_norm_typical, 20)
print(f"Typical feature norm: {feat_norm_typical:.2f}")
print(f"Alphas: from {alphas[0]:.2f} to {alphas[-1]:.2f}")

def _compute_adversarial():
    """Dla każdego kierunku i każdej alfy, prob_AI dla edytowanych feature vectors."""
    results = {}
    
    # Baseline — predykcje na oryginalnych feature vectors
    probs_ai_baseline = head_from_gap.predict(feats_ai_test, batch_size=256, verbose=0).flatten()
    probs_real_baseline = head_from_gap.predict(feats_real_test, batch_size=256, verbose=0).flatten()
    results['baseline'] = {
        'probs_ai': probs_ai_baseline,
        'probs_real': probs_real_baseline,
    }
    print(f"Baseline mean prob_AI on AI samples:   {probs_ai_baseline.mean():.4f}")
    print(f"Baseline mean prob_AI on Real samples: {probs_real_baseline.mean():.4f}")

    # Dla każdego kierunku — przesuń AI features w stronę Real (czyli ODEJMIJ kierunek AI)
    for method in ['mean', 'svm', 'lr']:
        direction = directions['gap'][method]
        probs_per_alpha = np.zeros((len(alphas), len(feats_ai_test)), dtype=np.float32)
        
        for i, alpha in enumerate(alphas):
            # Edycja: feature vector AI − α · kierunek_AI = przesunięcie ku Real
            edited = feats_ai_test - alpha * direction
            probs = head_from_gap.predict(edited, batch_size=256, verbose=0).flatten()
            probs_per_alpha[i] = probs
        
        results[method] = probs_per_alpha
        print(f"  Direction '{method}': probs at α=0: {probs_per_alpha[0].mean():.3f}, "
              f"at α=max: {probs_per_alpha[-1].mean():.3f}")
    
    return results

if ADVERSARIAL_PATH.exists():
    print("  ↳ Loading adversarial results from cache...")
    adv_data = dict(np.load(ADVERSARIAL_PATH, allow_pickle=True))
    # numpy savez wraps dicts as 0-d arrays — unwrap
    adv_results = {}
    for k in ['mean', 'svm', 'lr']:
        adv_results[k] = adv_data[k]
    adv_results['baseline'] = adv_data['baseline'].item()
else:
    adv_results = _compute_adversarial()
    np.savez_compressed(
        ADVERSARIAL_PATH,
        mean=adv_results['mean'],
        svm=adv_results['svm'],
        lr=adv_results['lr'],
        baseline=np.array(adv_results['baseline'], dtype=object),
        alphas=alphas,
    )
    print("  ✅ Saved")


In [ ]:
# ─── Wizualizacja: prob_AI(α) dla każdej metody ───

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Lewy wykres: średnia prob_AI vs α dla 3 metod
ax = axes[0]
colors = {'mean': '#1f77b4', 'svm': '#ff7f0e', 'lr': '#2ca02c'}
for method in ['mean', 'svm', 'lr']:
    probs = adv_results[method]  # shape (n_alphas, n_samples)
    mean_prob = probs.mean(axis=1)
    std_prob = probs.std(axis=1)
    ax.plot(alphas, mean_prob, color=colors[method], lw=2, label=f"{method.upper()}")
    ax.fill_between(alphas, mean_prob - std_prob, mean_prob + std_prob,
                    color=colors[method], alpha=0.15)

ax.axhline(0.5, color='black', linestyle='--', lw=1, alpha=0.5, label='decision threshold')
ax.set_xlabel('α (przesunięcie w przeciwną stronę kierunku AI)')
ax.set_ylabel('Mean prob_AI on edited features')
ax.set_title('Adversarial test: czy kierunek steruje decyzją?')
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Prawy wykres: % próbek przefliplowanych z AI na Real (prob < 0.5)
ax = axes[1]
for method in ['mean', 'svm', 'lr']:
    probs = adv_results[method]
    flip_rate = (probs < 0.5).mean(axis=1) * 100
    ax.plot(alphas, flip_rate, color=colors[method], lw=2, label=f"{method.upper()}")
ax.set_xlabel('α')
ax.set_ylabel('% próbek flipped do "Real"')
ax.set_title('Skuteczność przesunięcia: % AI sklasyfikowanych jako Real')
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(-5, 105)

plt.tight_layout()
plt.savefig(CACHE_DIR / 'adversarial_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("INTERPRETACJA WYNIKÓW:")
print("="*60)
print("✅ DOBRE: krzywe prob_AI monotonicznie spadają, sięgają < 0.5")
print("✅ DOBRE: flip rate sięga >50% przy umiarkowanej α")
print("⚠ OSTROŻNIE: jeśli krzywe są płaskie — kierunek jest słaby")
print("⚠ OSTROŻNIE: jeśli flip rate skacze do 100% przy bardzo małym α —")
print("     model jest bardzo wrażliwy w pobliżu granicy decyzji")
print("     (możliwe adversarial leakage zamiast prawdziwej zmiany)")


## 7. Per-generator analiza — czy SD/DALL-E/Midjourney mają wspólny kierunek AI?

Dla Twojego use case'u ("dziwne cienie i odbicia") to jest kluczowe pytanie. Jeśli kierunki AI różnią się między generatorami, to:
- Każdy generator ma **własną sygnaturę artefaktów**
- Wektor uśredniony "AI-ness" jest tylko aproksymacją
- Dla precyzyjnej edycji warto użyć kierunku per-generator, nie zbiorczego

Liczymy `μ_generator − μ_real` dla każdego generatora osobno, normalizujemy, i porównujemy cosinusami.


In [ ]:
PER_GEN_PATH = CACHE_DIR / "per_generator_directions.pkl"

def _compute_per_generator():
    """Dla każdego generatora osobno — kierunek od Real do tego generatora."""
    result = {}
    
    for space_name, feats in [('gap', features_gap), ('dense', features_dense)]:
        mu_real = feats[label_a == 0].mean(axis=0)
        per_gen = {}
        
        for k in range(1, 6):  # generators 1-5 (0 = Real)
            mask = label_b == k
            if mask.sum() < 100:
                continue
            mu_gen = feats[mask].mean(axis=0)
            direction = mu_gen - mu_real
            per_gen[k] = {
                'name': LABEL_B_MAP[k],
                'direction': direction / np.linalg.norm(direction),
                'mu_gen': mu_gen,
                'count': int(mask.sum()),
            }
        
        result[space_name] = {
            'mu_real': mu_real,
            'generators': per_gen,
        }
    
    return result

per_gen_data = cache_pickle(PER_GEN_PATH, _compute_per_generator)


In [ ]:
# ─── Macierz cosine similarity między kierunkami per-generator ───

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, space_name in enumerate(['gap', 'dense']):
    gens = per_gen_data[space_name]['generators']
    gen_keys = sorted(gens.keys())
    n = len(gen_keys)
    
    cos_matrix = np.zeros((n, n))
    for i, ki in enumerate(gen_keys):
        for j, kj in enumerate(gen_keys):
            cos_matrix[i, j] = np.dot(gens[ki]['direction'], gens[kj]['direction'])
    
    ax = axes[ax_idx]
    im = ax.imshow(cos_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    
    labels_short = [gens[k]['name'] for k in gen_keys]
    ax.set_xticks(range(n)); ax.set_xticklabels(labels_short, rotation=45, ha='right')
    ax.set_yticks(range(n)); ax.set_yticklabels(labels_short)
    
    for i in range(n):
        for j in range(n):
            color = 'white' if abs(cos_matrix[i, j]) > 0.5 else 'black'
            ax.text(j, i, f'{cos_matrix[i, j]:.2f}', ha='center', va='center',
                    color=color, fontsize=10)
    
    ax.set_title(f'{space_name.upper()} — cosine similarity między kierunkami')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(CACHE_DIR / 'per_generator_cosine_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Wypisz najważniejsze obserwacje
print("\nINTERPRETACJA:")
print("="*60)
print("• cos ≈ 1.0  → generatory mają BARDZO PODOBNE artefakty")
print("  → wspólny kierunek 'AI' jest sensowny")
print("• cos ≈ 0.5  → częściowe pokrywanie się sygnatur")
print("• cos ≈ 0.0  → generatory mają RÓŻNE typy artefaktów")
print("  → trzeba edytować osobno per-generator")
print("• cos < 0    → kierunki są przeciwstawne (rzadkie, podejrzane)")


In [ ]:
# ─── Bonus: który generator jest 'najdalej' od Real? ───
# Norma wektora μ_gen − μ_real mówi nam o łatwości detekcji

print("Odległość średniego feature vectora każdego generatora od Real (GAP space):")
print("-" * 60)
gens = per_gen_data['gap']['generators']
mu_real = per_gen_data['gap']['mu_real']

distances = []
for k in sorted(gens.keys()):
    g = gens[k]
    dist = np.linalg.norm(g['mu_gen'] - mu_real)
    distances.append((g['name'], dist, g['count']))

distances.sort(key=lambda x: x[1], reverse=True)

for name, dist, count in distances:
    bar = '█' * int(dist / max(d[1] for d in distances) * 30)
    print(f"  {name:<20} {bar:<30}  {dist:.2f}  (n={count:,})")

print("\nNajdalej od Real → najłatwiej wykrywalne. Najbliżej → najtrudniej.")


## 8. Podsumowanie — czy idziemy do Fazy 2?

Decyzja o Fazie 2 zależy od tego, co tu zobaczyliśmy. Cztery scenariusze:

| Wynik | Decyzja |
|---|---|
| d' > 2, flip rate sięga >70%, kierunki spójne (cos > 0.9) | **Idziemy do Fazy 2** — kierunek istnieje, ma sens, możemy budować VAE |
| d' 1-2, flip rate 30-70%, kierunki częściowo spójne | **Faza 2 z ostrożnością** — kierunek ogólny istnieje, ale rozważ per-generator |
| d' < 1, flip rate < 30% | **Stop** — przemyśl klasyfikator (może niedouczony) lub podejście |
| Wysoki flip rate przy bardzo małym α | **Adversarial leakage** — model wrażliwy lokalnie, ale Faza 2 może nie dać sensownych obrazów |

Konkretne wnioski wypisane poniżej (wartości z Twojego runu).


In [ ]:
# ─── Tabela podsumowująca ───

print("=" * 70)
print("PODSUMOWANIE FAZY 1")
print("=" * 70)

for space_name in ['gap', 'dense']:
    print(f"\n--- {space_name.upper()} space ---")
    d = directions[space_name]
    print(f"  SVM accuracy:           {d['svm_acc']:.4f}")
    print(f"  LR accuracy:            {d['lr_acc']:.4f}")
    print(f"  cosine(mean, svm):      {np.dot(d['mean'], d['svm']):+.4f}")
    print(f"  cosine(mean, lr):       {np.dot(d['mean'], d['lr']):+.4f}")
    print(f"  cosine(svm, lr):        {np.dot(d['svm'], d['lr']):+.4f}")

    # d' (Cohen's) dla każdej metody
    feats = features_gap if space_name == 'gap' else features_dense
    for method in ['mean', 'svm', 'lr']:
        proj = feats @ d[method]
        proj_real = proj[label_a == 0]
        proj_ai = proj[label_a == 1]
        d_prime = (proj_ai.mean() - proj_real.mean()) / np.sqrt(0.5 * (proj_ai.var() + proj_real.var()))
        print(f"  d' ({method}):              {d_prime:.3f}")

# Adversarial summary (na GAP space)
print(f"\n--- ADVERSARIAL TEST (GAP space) ---")
for method in ['mean', 'svm', 'lr']:
    probs = adv_results[method]
    final_flip_rate = (probs[-1] < 0.5).mean() * 100
    alpha_for_50 = None
    flip_rates = (probs < 0.5).mean(axis=1) * 100
    cross = np.where(flip_rates >= 50)[0]
    if len(cross) > 0:
        alpha_for_50 = alphas[cross[0]]
    print(f"  {method.upper():<6} final flip rate: {final_flip_rate:.1f}%   "
          f"α for 50% flip: {f'{alpha_for_50:.2f}' if alpha_for_50 else 'N/A'}")

print("\n" + "=" * 70)
print("NASTĘPNE KROKI:")
print("=" * 70)
print("""
1. Sprawdź wykresy w cache_phase1/:
   - umap_overview.png
   - projection_histograms.png
   - adversarial_curves.png
   - per_generator_cosine_matrix.png

2. Jeśli wyniki są pozytywne (d' > 1.5, flip rate > 50%):
   → Faza 2: trenowanie VAE z dekoderem (TF lub PyTorch)
   → Edycja będzie działać w nowym latent space VAE,
     ale można porównać kierunki AI-ness w obu przestrzeniach

3. Jeśli wyniki per-generator pokazują niespójność (cos < 0.7):
   → W Fazie 2 rozważ trening osobnego kierunku per generator
   → Albo użyj 'common direction' (np. SVM trenowany binarnie) jako
     bazy, plus per-generator residuum jako fine-tuning

4. Wszystkie cache pliki w cache_phase1/ są przenośne między sesjami.
   Ten notebook może być teraz uruchomiony w sekundach.
""")


In [ ]:
# ─── Inwentaryzacja cache ───
# Co dostałeś jako artefakty wielokrotnego użytku?

import os
print("Zawartość cache_phase1/:")
print("-" * 60)
total_size = 0
for f in sorted(CACHE_DIR.iterdir()):
    size = os.path.getsize(f)
    total_size += size
    size_str = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
    print(f"  {f.name:<40} {size_str:>10}")
print("-" * 60)
print(f"  TOTAL: {total_size/1e6:.1f} MB")

print("""
Te pliki zawierają:
  features_gap.npy           — feature vectors całego datasetu (1280D)
  features_dense.npy         — feature vectors całego datasetu (256D)
  labels.npz                 — etykiety Label_A i Label_B
  ai_directions.pkl          — kierunki AI dla obu warstw, 3 metody
  sklearn_models.pkl         — wytrenowane SVM i LR
  umap_*_embedding.npy       — 2D embeddings
  umap_*_reducer.pkl         — reducer UMAP (do projekcji nowych próbek)
  per_generator_directions.pkl — kierunki per generator
  adversarial_test.npz       — wyniki testu adversarial

Możesz załadować dowolny z tych plików w innym notebooku
i kontynuować analizę bez ponownego liczenia features.
""")
